Trae los activos en tendencia alcista que esten por dar entrada segun índicador MACD.

In [1]:
# Importación.

from datos import obtener_tickers 
import pandas_ta as ta
import pandas as pd

%load_ext autoreload
%autoreload 2

In [ ]:
# Descarga de datos temporalidad diaria.

data_1d = obtener_tickers("1d", "3y")

Se han cargado 273 tickers...


In [ ]:
# ANÁLISIS TÉCNICO 1D - Indicador MACD 12 - 26 - 9
empresas_diario = []

print(f"Iniciando análisis técnico para {len(data_1d.columns.get_level_values(0).unique())} activos...")

for ticker in data_1d.columns.get_level_values(0).unique():
    try:
        df = data_1d.xs(ticker, axis=1, level=0).dropna()
        
        if len(df) < 260:
            continue

        df.ta.ema(length=150, close='Close', append=True)
        df.ta.ema(length=200, close='Close', append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        precio = df['Close'].iloc[-1]
        ema_150 = df['EMA_150'].iloc[-1]
        ema_200 = df['EMA_200'].iloc[-1]
        
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        macd_ant = df['MACD_12_26_9'].iloc[-2]
        sig_ant = df['MACDs_12_26_9'].iloc[-2]

        tendencia_alcista = (precio > ema_150) and (precio > ema_200)
        macd_bajo_cero = macd_act < 0
        cruce_o_punto = (macd_act >= sig_act) and (macd_ant < sig_ant)

        # Cálculo de porcentajes respecto a las EMAs
        pct_vs_ema150 = ((precio - ema_150) / ema_150) * 100
        pct_vs_ema200 = ((precio - ema_200) / ema_200) * 100

        print(f"[{ticker}] P: {precio:.2f} | EMA150: {ema_150:.2f} ({pct_vs_ema150:+.2f}%) | EMA200: {ema_200:.2f} ({pct_vs_ema200:+.2f}%) | MACD: {macd_act:.3f} | Señal: {sig_act:.3f}")

        if tendencia_alcista and macd_bajo_cero and cruce_o_punto:
            print(f" ¡MATCH ENCONTRADO!: {ticker}")
            empresas_diario.append({
                'Ticker': ticker,
                'Close': round(precio, 2),
                'MACD': round(macd_act, 3),
                'Senal': round(sig_act, 3),
                'EMA150': round(ema_150, 2),
                'EMA200': round(ema_200, 2),
                'Pct_vs_EMA150': round(pct_vs_ema150, 2),
                'Pct_vs_EMA200': round(pct_vs_ema200, 2)
            })
            
    except Exception as e:
        print(f"Error procesando {ticker}: {e}")
        continue

if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print("\n" + "=" * 90)
    print("PAPELES DETECTADOS EN DIARIO (MACD BAJO CERO + CRUCE + TENDENCIA EMA):")
    print(df_final.to_string(index=False))
    print("=" * 90)
else:
    print("\nSin señales diarias encontradas bajo los criterios establecidos.")

Iniciando análisis técnico para 539 activos...
[AMZN] P: 251.71 | EMA150: 248.94 (+1.11%) | EMA200: 245.73 (+2.43%) | MACD: -1.784 | Señal: -0.745
[APH] P: 82.03 | EMA150: 77.36 (+6.03%) | EMA200: 75.07 (+9.27%) | MACD: -0.839 | Señal: -0.983
 ¡MATCH ENCONTRADO!: APH
[MCHP] P: 78.89 | EMA150: 77.68 (+1.56%) | EMA200: 76.81 (+2.71%) | MACD: 0.778 | Señal: 0.358
[HAS] P: 119.10 | EMA150: 92.58 (+28.64%) | EMA200: 90.41 (+31.74%) | MACD: 7.556 | Señal: 6.338
[VST] P: 137.25 | EMA150: 151.05 (-9.14%) | EMA200: 152.85 (-10.21%) | MACD: -2.065 | Señal: -0.277
[CRL] P: 348.43 | EMA150: 255.47 (+36.39%) | EMA200: 241.64 (+44.19%) | MACD: 19.184 | Señal: 16.230
[REGN] P: 968.99 | EMA150: 749.56 (+29.27%) | EMA200: 738.85 (+31.15%) | MACD: 31.247 | Señal: 13.901
[PEP] P: 139.29 | EMA150: 137.99 (+0.94%) | EMA200: 139.14 (+0.11%) | MACD: 1.553 | Señal: 0.718
[URA] P: 34.11 | EMA150: 43.07 (-20.81%) | EMA200: 43.50 (-21.58%) | MACD: -1.663 | Señal: -1.755
[CMI] P: 506.87 | EMA150: 572.49 (-11.46%)

In [ ]:
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print("\n" + "=" * 50)
    print("PAPELES DETECTADOS EN DIARIO:")
    print(df_final.to_string(index=False))
    print("=" * 50)
else:
    print("Sin señales diarias encontradas.")
    
print(f"Última barra: {df.index[-1]} | Barras: {len(df)}")



PAPELES DETECTADOS EN DIARIO:
Ticker   Close    MACD   Senal  EMA150  EMA200  Pct_vs_EMA150  Pct_vs_EMA200
   APH   82.03  -0.839  -0.983   77.36   75.07           6.03           9.27
   XYZ   79.05  -0.099  -0.245   75.11   74.03           5.24           6.77
   WAB  286.46  -1.154  -1.332  272.96  265.79           4.95           7.78
   TGT  156.39  -0.776  -1.037  141.26  136.11          10.71          14.90
  DXCM   84.23  -1.364  -1.478   79.11   77.81           6.48           8.25
  MPWR 1295.36 -25.328 -25.840 1289.90 1259.05           0.42           2.88
Última barra: 2026-11-02 00:00:00 | Barras: 752
